# Continuous MES, forward-anchored

Your check of step 1.7: one year of continuous MES with its adjusted close, its raw close, and the offset between them. At each roll the raw close jumps by the gap between the two contracts, the offset steps by that gap, and the adjusted close runs on without a jump.

Set `YEAR` in the next cell, pick the `.venv` kernel, and run all cells. The roll calendar is built from every MES bar in your store, so the offsets are those of the whole history since the product's launch. The charts show one point per trade date: its last 1-minute close. Prices print in index points.

In [ ]:
import os
from datetime import date

import polars as pl
from IPython.display import SVG, display

from fefat.config import resolve_paths
from fefat.market import instruments
from fefat.market.builtin_calendars import BUILT_IN
from fefat.market.calendar import sessions_of
from fefat.market.continuous import volume_rolls
from fefat.market.layout import Dataset, PartitionRequest
from fefat.market.schema import parse_schema_name
from fefat.market.store import Store

ROOT = "MES"
YEAR = 2025

paths = resolve_paths(os.environ)
store = Store(paths.data_dir)
instrument = instruments.get(ROOT)
sessions = sessions_of(BUILT_IN[instrument.calendar_id], paths.cache_dir)
dataset = Dataset(provider="databento", schema_name=parse_schema_name("ohlcv-1m"), root=ROOT)
request = PartitionRequest(dataset=dataset, start=instrument.valid_from, end=date(YEAR, 12, 31))
bars = store.scan_contracts(request).collect()
calendar = volume_rolls(bars, sessions, instrument, bars.get_column("ts_close").max())
calendar

In [ ]:
POINTS = 1_000_000_000
of_the_year = pl.col("trade_date").dt.year() == YEAR
daily = (
    calendar.continuous_bars(bars)
    .filter(of_the_year)
    .group_by("trade_date")
    .agg(pl.all().sort_by("ts_open").last())
    .sort("trade_date")
    .select(
        "trade_date",
        "contract",
        close=pl.col("close") / POINTS,
        close_raw=pl.col("close_raw") / POINTS,
        adj_offset=pl.col("adj_offset") / POINTS,
    )
)
rolls = calendar.rolls().filter(of_the_year)
rolls.select("trade_date", "rolled_from", "contract", gap=pl.col("gap") / POINTS, reason="reason")

In [ ]:
COLORS = ("#1f6feb", "#cf222e", "#1a7f37")


def line_chart(title, days, lines, marks, width=960, height=280):
    """An SVG chart of each line (a name and a value for each day), with a dashed line at each
    day of `marks`."""
    values = [value for line in lines.values() for value in line]
    low, high = min(values), max(values)

    def x(index):
        return 70 + index * (width - 90) / max(len(days) - 1, 1)

    def y(value):
        return height - 30 - (value - low) * (height - 70) / ((high - low) or 1)

    text = 'font-family="sans-serif" font-size="12"'
    parts = [
        f'<text x="70" y="18" {text} font-weight="bold">{title}</text>',
        f'<text x="4" y="{y(high) + 4:.0f}" {text}>{high:,.2f}</text>',
        f'<text x="4" y="{y(low) + 4:.0f}" {text}>{low:,.2f}</text>',
        f'<text x="70" y="{height - 8}" {text}>{days[0]}</text>',
        f'<text x="{width - 90}" y="{height - 8}" {text}>{days[-1]}</text>',
    ]
    for day in marks:
        at = x(days.index(day))
        parts.append(
            f'<line x1="{at:.1f}" y1="30" x2="{at:.1f}" y2="{height - 30}" '
            'stroke="#8c959f" stroke-dasharray="4 3"/>'
        )
    for order, (name, line) in enumerate(lines.items()):
        points = " ".join(f"{x(index):.1f},{y(value):.1f}" for index, value in enumerate(line))
        parts.append(
            f'<polyline fill="none" stroke="{COLORS[order]}" stroke-width="1.5" points="{points}"/>'
        )
        parts.append(
            f'<text x="{width - 330 + 160 * order}" y="18" {text} fill="{COLORS[order]}">{name}</text>'
        )
    body = "".join(parts)
    return SVG(
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">{body}</svg>'
    )


days = daily.get_column("trade_date").to_list()
roll_days = [day for day in rolls.get_column("trade_date").to_list() if day in days]
display(
    line_chart(
        f"{ROOT} {YEAR}: daily close, adjusted and raw (dashed: rolls)",
        days,
        {
            "close (adjusted)": daily.get_column("close").to_list(),
            "close_raw": daily.get_column("close_raw").to_list(),
        },
        roll_days,
    )
)
display(
    line_chart(
        f"{ROOT} {YEAR}: adj_offset in points",
        days,
        {"adj_offset": daily.get_column("adj_offset").to_list()},
        roll_days,
    )
)

## What to look for

- At each dashed line, the red raw close jumps to the new contract's price, while the blue adjusted close carries on from the day before without that jump.
- The green offset is flat between rolls and steps at each dashed line by minus the roll's gap, as the table above lists it.
- Between two rolls the adjusted and raw closes run parallel, a constant offset apart.